# mosaic-foundations

_Investigation `mosaic-foundations` — coder reproduction notebook._

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/mosaic-compose/mosaic-compose').is_dir():
    REPO = Path('/home/runner/work/mosaic-compose/mosaic-compose')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from mosaic_compose.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: `mosaic_dfba_lactobacillus`


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mosaic_dfba_lactobacillus` | `mosaic_compose.composites.dfba.mosaic_dfba_lactobacillus` | 60 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `mosaic_compose.composites.dfba.mosaic_dfba_lactobacillus`** — `spec_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus` (materialized from its `@composite_generator`)


In [ ]:
spec_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus = materialize_composite('mosaic_compose.composites.dfba.mosaic_dfba_lactobacillus')
describe_spec(spec_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus = dict(spec_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus['_overrides'])
import json as _json; print(_json.dumps(ov_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus, indent=2, default=str))
# e.g.:  ov_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mosaic_dfba_lactobacillus ===
STUDY = 'mosaic_dfba_lactobacillus'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_mosaic_dfba_lactobacillus = 60
INTERVAL_mosaic_dfba_lactobacillus = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus if 'ov_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus' in dir() else None  # edits from the panel cell, if any
            spec_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus = materialize_composite('mosaic_compose.composites.dfba.mosaic_dfba_lactobacillus', _ov)
            _comp = Composite({'state': spec_mosaic_compose_composites_dfba_mosaic_dfba_lactobacillus['state']}, core=core)
            _comp.run(STEPS_mosaic_dfba_lactobacillus)
        print('ran', 'mosaic_dfba_lactobacillus', f'(mosaic_compose.composites.dfba.mosaic_dfba_lactobacillus) for', STEPS_mosaic_dfba_lactobacillus, 'steps')
    except Exception as _e:
        print(f"mosaic_dfba_lactobacillus not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**dfba_growth**


In [ ]:
# dfba_growth
show_viz(_render_one('local:DFBAGrowth', {'inputs_map': {'biomass': 'biomass', 'glucose': 'substrates.glucose', 'glutamate': 'substrates.glutamate', 'time': 'global_time'}}, RUNS_DB, STUDY_YAML))

## Study: `mosaic_glv_vaginal`


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mosaic_glv_vaginal` | `mosaic_compose.composites.community.mosaic_glv_vaginal` | 120 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `mosaic_compose.composites.community.mosaic_glv_vaginal`** — `spec_mosaic_compose_composites_community_mosaic_glv_vaginal` (materialized from its `@composite_generator`)


In [ ]:
spec_mosaic_compose_composites_community_mosaic_glv_vaginal = materialize_composite('mosaic_compose.composites.community.mosaic_glv_vaginal')
describe_spec(spec_mosaic_compose_composites_community_mosaic_glv_vaginal)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_mosaic_compose_composites_community_mosaic_glv_vaginal maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_mosaic_compose_composites_community_mosaic_glv_vaginal = dict(spec_mosaic_compose_composites_community_mosaic_glv_vaginal['_overrides'])
import json as _json; print(_json.dumps(ov_mosaic_compose_composites_community_mosaic_glv_vaginal, indent=2, default=str))
# e.g.:  ov_mosaic_compose_composites_community_mosaic_glv_vaginal['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mosaic_glv_vaginal ===
STUDY = 'mosaic_glv_vaginal'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_mosaic_glv_vaginal = 120
INTERVAL_mosaic_glv_vaginal = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_mosaic_compose_composites_community_mosaic_glv_vaginal if 'ov_mosaic_compose_composites_community_mosaic_glv_vaginal' in dir() else None  # edits from the panel cell, if any
            spec_mosaic_compose_composites_community_mosaic_glv_vaginal = materialize_composite('mosaic_compose.composites.community.mosaic_glv_vaginal', _ov)
            _comp = Composite({'state': spec_mosaic_compose_composites_community_mosaic_glv_vaginal['state']}, core=core)
            _comp.run(STEPS_mosaic_glv_vaginal)
        print('ran', 'mosaic_glv_vaginal', f'(mosaic_compose.composites.community.mosaic_glv_vaginal) for', STEPS_mosaic_glv_vaginal, 'steps')
    except Exception as _e:
        print(f"mosaic_glv_vaginal not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**glv_competition**


In [ ]:
# glv_competition
show_viz(_render_one('local:GLVCompetition', {'inputs_map': {'lactobacillus': 'stores.species.Lactobacillus', 'gardnerella': 'stores.species.Gardnerella', 'time': 'global_time'}}, RUNS_DB, STUDY_YAML))

**glv_phase_portrait**


In [ ]:
# glv_phase_portrait
show_viz(_render_one('local:GLVPhasePortrait', {'inputs_map': {'lactobacillus': 'stores.species.Lactobacillus', 'gardnerella': 'stores.species.Gardnerella'}}, RUNS_DB, STUDY_YAML))

## Study: `mosaic_estrogen_enterohepatic`


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mosaic_estrogen_enterohepatic` | `mosaic_compose.composites.hormone.mosaic_estrogen_enterohepatic` | 120 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `mosaic_compose.composites.hormone.mosaic_estrogen_enterohepatic`** — `spec_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic` (materialized from its `@composite_generator`)


In [ ]:
spec_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic = materialize_composite('mosaic_compose.composites.hormone.mosaic_estrogen_enterohepatic')
describe_spec(spec_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic = dict(spec_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic['_overrides'])
import json as _json; print(_json.dumps(ov_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic, indent=2, default=str))
# e.g.:  ov_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mosaic_estrogen_enterohepatic ===
STUDY = 'mosaic_estrogen_enterohepatic'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_mosaic_estrogen_enterohepatic = 120
INTERVAL_mosaic_estrogen_enterohepatic = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic if 'ov_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic' in dir() else None  # edits from the panel cell, if any
            spec_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic = materialize_composite('mosaic_compose.composites.hormone.mosaic_estrogen_enterohepatic', _ov)
            _comp = Composite({'state': spec_mosaic_compose_composites_hormone_mosaic_estrogen_enterohepatic['state']}, core=core)
            _comp.run(STEPS_mosaic_estrogen_enterohepatic)
        print('ran', 'mosaic_estrogen_enterohepatic', f'(mosaic_compose.composites.hormone.mosaic_estrogen_enterohepatic) for', STEPS_mosaic_estrogen_enterohepatic, 'steps')
    except Exception as _e:
        print(f"mosaic_estrogen_enterohepatic not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**estrogen_enterohepatic**


In [ ]:
# estrogen_enterohepatic
show_viz(_render_one('local:EstrogenEnterohepatic', {'inputs_map': {'e_blood': 'stores.species.E_blood', 'eg_gut': 'stores.species.EG_gut', 'e_gut': 'stores.species.E_gut', 'time': 'global_time'}}, RUNS_DB, STUDY_YAML))

## Study: `mosaic_dfba_coculture`


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mosaic_dfba_coculture` | `mosaic_compose.composites.coculture.mosaic_dfba_coculture` | 50 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `mosaic_compose.composites.coculture.mosaic_dfba_coculture`** — `spec_mosaic_compose_composites_coculture_mosaic_dfba_coculture` (materialized from its `@composite_generator`)


In [ ]:
spec_mosaic_compose_composites_coculture_mosaic_dfba_coculture = materialize_composite('mosaic_compose.composites.coculture.mosaic_dfba_coculture')
describe_spec(spec_mosaic_compose_composites_coculture_mosaic_dfba_coculture)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_mosaic_compose_composites_coculture_mosaic_dfba_coculture maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_mosaic_compose_composites_coculture_mosaic_dfba_coculture = dict(spec_mosaic_compose_composites_coculture_mosaic_dfba_coculture['_overrides'])
import json as _json; print(_json.dumps(ov_mosaic_compose_composites_coculture_mosaic_dfba_coculture, indent=2, default=str))
# e.g.:  ov_mosaic_compose_composites_coculture_mosaic_dfba_coculture['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mosaic_dfba_coculture ===
STUDY = 'mosaic_dfba_coculture'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_mosaic_dfba_coculture = 50
INTERVAL_mosaic_dfba_coculture = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_mosaic_compose_composites_coculture_mosaic_dfba_coculture if 'ov_mosaic_compose_composites_coculture_mosaic_dfba_coculture' in dir() else None  # edits from the panel cell, if any
            spec_mosaic_compose_composites_coculture_mosaic_dfba_coculture = materialize_composite('mosaic_compose.composites.coculture.mosaic_dfba_coculture', _ov)
            _comp = Composite({'state': spec_mosaic_compose_composites_coculture_mosaic_dfba_coculture['state']}, core=core)
            _comp.run(STEPS_mosaic_dfba_coculture)
        print('ran', 'mosaic_dfba_coculture', f'(mosaic_compose.composites.coculture.mosaic_dfba_coculture) for', STEPS_mosaic_dfba_coculture, 'steps')
    except Exception as _e:
        print(f"mosaic_dfba_coculture not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coculture_competition**


In [ ]:
# coculture_competition
show_viz(_render_one('local:CocultureCompetition', {'inputs_map': {'biomass_ecoli': 'biomass_ecoli', 'biomass_lacto': 'biomass_lacto', 'glucose': 'substrates.glucose', 'time': 'global_time'}}, RUNS_DB, STUDY_YAML))

## Study: `mosaic_biofilm_colony`


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mosaic_biofilm_colony` | `mosaic_compose.composites.colony.mosaic_biofilm_colony` | 10800 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `mosaic_compose.composites.colony.mosaic_biofilm_colony`** — `spec_mosaic_compose_composites_colony_mosaic_biofilm_colony` (materialized from its `@composite_generator`)


In [ ]:
spec_mosaic_compose_composites_colony_mosaic_biofilm_colony = materialize_composite('mosaic_compose.composites.colony.mosaic_biofilm_colony')
describe_spec(spec_mosaic_compose_composites_colony_mosaic_biofilm_colony)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_mosaic_compose_composites_colony_mosaic_biofilm_colony maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_mosaic_compose_composites_colony_mosaic_biofilm_colony = dict(spec_mosaic_compose_composites_colony_mosaic_biofilm_colony['_overrides'])
import json as _json; print(_json.dumps(ov_mosaic_compose_composites_colony_mosaic_biofilm_colony, indent=2, default=str))
# e.g.:  ov_mosaic_compose_composites_colony_mosaic_biofilm_colony['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mosaic_biofilm_colony ===
STUDY = 'mosaic_biofilm_colony'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_mosaic_biofilm_colony = 10800
INTERVAL_mosaic_biofilm_colony = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_mosaic_compose_composites_colony_mosaic_biofilm_colony if 'ov_mosaic_compose_composites_colony_mosaic_biofilm_colony' in dir() else None  # edits from the panel cell, if any
            spec_mosaic_compose_composites_colony_mosaic_biofilm_colony = materialize_composite('mosaic_compose.composites.colony.mosaic_biofilm_colony', _ov)
            _comp = Composite({'state': spec_mosaic_compose_composites_colony_mosaic_biofilm_colony['state']}, core=core)
            _comp.run(STEPS_mosaic_biofilm_colony)
        print('ran', 'mosaic_biofilm_colony', f'(mosaic_compose.composites.colony.mosaic_biofilm_colony) for', STEPS_mosaic_biofilm_colony, 'steps')
    except Exception as _e:
        print(f"mosaic_biofilm_colony not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**biofilm_growth**


In [ ]:
# biofilm_growth
show_viz(_render_one('local:BiofilmColonyGrowth', {'inputs_map': {'cell_count': 'cell_count', 'time': 'global_time'}}, RUNS_DB, STUDY_YAML))